In [ ]:
import pandas as pd

# wczytuje oryginalny plik
df = pd.read_csv("open_library_articles.csv")

print(f"Przed czyszczeniem: {len(df)} rekordów")

# napawia rozbite wiersze (gdy title ma \n + spacje)
df['title'] = df['title'].str.replace(r'\s+\n\s+', ' ', regex=True).str.strip()
df['description'] = df['description'].str.replace(r'\s+\n\s+', ' ', regex=True).str.strip()

# usuwa dokładne duplikaty (gdy wszystkie kolumny identyczne)
df.drop_duplicates(inplace=True)

# usuwa duplikaty po tytule + autorach ### najpierw czyszczenie authors z pustych i spacji
df['authors_clean'] = df['authors'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()

df.drop_duplicates(subset=['title', 'authors_clean'], keep='first', inplace=True)
df.drop(columns=['authors_clean'], inplace=True)  ### niepotrzebna po już do usunięcia

# dodatkowe czyszczenie
df['title'] = df['title'].str.strip('"')  # usuwa cudzysłowy z początku/końca
df['publish_year'] = pd.to_numeric(df['publish_year'], errors='coerce')  # poprawia lata

# sortowanie
df.sort_values(['matched_keyword', 'publish_year', 'title'], inplace=True)
df.reset_index(drop=True, inplace=True)

# zapis fixa
output_file = "open_library_articles_fixed.csv"
df.to_csv(output_file, index=False, encoding='utf-8')

print(f"Po czyszczeniu: {len(df)} rekordów (usunięto duplikaty i śmieci)")
print(f"Zapisano do: {output_file}")
print(f"Przykładowe poprawione wiersze:")
print(df[['matched_keyword', 'title', 'authors', 'publish_year']].head(10))